# CS 3892 / 5892 — Session 8 · SMT, Theories, and Bounded Reachability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-22-smt-theories-and-bounded-reachability.ipynb)

**Tuesday, September 22, 2026.** How a theory plugs into the boolean search, and
how a *system* — something that takes steps — becomes one formula a solver can
decide.

**The one move today:** unroll the transition relation `k` times, assert the
property is broken somewhere, and ask. `sat` hands you the counterexample trace.

> **Demo 2 is the method for HW1 Part 3**, on a deliberately different machine.
> Copy the method, not the number — finding the threshold is what Part 3 asks.


## Setup

Run this once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-22-smt-theories-and-bounded-reachability"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

## 1. Why a SAT solver alone is not enough

The **boolean skeleton** of an SMT formula is what a SAT solver sees: every
atom becomes an opaque boolean. For `(x > 5) ∧ (x < 3)` the skeleton is `p ∧ q`,
which is satisfiable. The formula is not.

The **theory solver** closes that gap. The SAT solver proposes `p = q = True`;
the LIA theory solver checks that assignment against what the symbols *mean*,
finds it impossible, and returns the lemma `¬(p ∧ q)`. The SAT solver adds it as
a clause and never proposes that assignment again.

That loop is **DPLL(T)**.

In [ ]:
run(SM  / "01_theory_conflict.smt2")
run(PYD / "01_theory_conflict.py")

## 2. Bounded reachability — a system becomes one formula

Thursday closed on the question: *a solver decides one fixed formula, but a
system has runs — so how do you turn "this never goes wrong" into one formula?*

Here is the answer, and it is four lines:

```
x0 = init                                  the initial state
T(x0,x1) ∧ T(x1,x2) ∧ … ∧ T(x[k-1],xk)     the transition relation, k times
bad(x0) ∨ bad(x1) ∨ … ∨ bad(xk)            the NEGATED property
check-sat
```

`sat` means **reachable**, and the model **is** the counterexample trace.
`unsat` means no counterexample **within k steps** — and nothing at all about
step `k+1`.

The machine below is a thermostat, **not** HW1's counter. Run both `.smt2` files
and compare them: `k=4` finds the bug, `k=3` reports `unsat` in exactly the words
a safe system would use.

In [ ]:
run(PYD / "02_bmc_unroll.py")

In [ ]:
run(SM / "02_bmc_unroll_k4.smt2")
run(SM / "03_bmc_unroll_k3.smt2")

## 3. 2ⁿ is the worst case, not the case

Thursday's slide said SAT is NP-complete and the worst case is `2ⁿ`. True — and
not what you see. This is the scaling demo promised in class.

An implication chain with **100,000 variables** goes through in a fraction of a
second, because unit propagation alone decides it. **Pigeonhole** — `n+1` pigeons
into `n` holes — falls over around `n = 10`, and that is a *theorem* about
resolution (Haken, 1985), not a solver limitation.

Which instance you have matters more than how big it is. **This is also HW1
Part 2**: report the times, and say where the wall is.

In [ ]:
run(PYD / "03_scaling.py")

## 4. The abstract UNSAT core

Thursday's core example was the leave policy, which carries meaning with it. He
offered a version with no meaning at all, so the mechanism is visible.

Five assertions over four unrelated booleans. Three of them are the reason for
`unsat`; the other two are innocent and the core leaves them out.

In a real query the assertions are your **specification**, and the core answers
*which requirements contradict each other* — far more useful than the word
`unsat`.

In [ ]:
run(PYD / "04_abstract_core.py")

## Where this goes

| | |
|---|---|
| **Thursday Sep 24** | Transition systems properly — traces, reachability, and *inductive* invariants: how you prove safety for **all** `k` rather than one `k` at a time. |
| **HW1, due Thursday** | Part 2 is §3 above; Part 3 is §2 above on a different machine. |
| **HW2 (nuXmv)** | The same property, unbounded, by BDD reachability. |

Every file here: `github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-22-smt-theories-and-bounded-reachability`
